In [1]:
from interpret import show
import time
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV

# Load datasets

In [2]:
from dataLoader import load_msi_data, MSI_SATELLITE_TEST_DIR, MSI_SATELLITE_TRAIN_DIR, GT_TRAIN_CSV_PATH, column_names

gt_train_df = pd.read_csv(GT_TRAIN_CSV_PATH)

X_train_all = load_msi_data(MSI_SATELLITE_TRAIN_DIR)
X_test_all = load_msi_data(MSI_SATELLITE_TEST_DIR)
y_train_all = gt_train_df[column_names].values


In [3]:
from sklearn.preprocessing import StandardScaler

# TODO: Normalisation
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(X_train_all, y_train_all, test_size=0.8)


X_train = x_scaler.fit_transform(X_train)  # compute mean/std from train
X_test = x_scaler.transform(X_test)        # apply same scaling

y_train = y_scaler.fit_transform(y_train)
y_test = y_scaler.transform(y_test)

In [4]:
import numpy as np
from BaselineRegressor import BaselineRegressor

# Fit the baseline model
baseline_reg = BaselineRegressor()
baseline_reg = baseline_reg.fit(X_train, y_train)
baseline_predictions = baseline_reg.predict(X_test)

# Generate baseline values to be used in score computation
baselines = np.mean((y_test - baseline_predictions) ** 2, axis=0)


# Generate predictions slightly different from baseline predictions
np.random.seed(0)
predictions = np.zeros_like(y_test)
for column_index in range(predictions.shape[1]):
    class_mean_value = baseline_reg.mean[column_index]
    predictions[:, column_index] = np.random.uniform(low=class_mean_value - class_mean_value * 0.2,
                                                     high=class_mean_value + class_mean_value * 0.2,
                                                     size=len(predictions))

# Calculate MSE for each class
mse = np.mean((y_test - predictions) ** 2, axis=0)

# Calculate the score for each class individually
scores = mse / baselines

# Calculate the final score
final_score = np.mean(scores)

for score, class_name in zip(scores, column_names):
    print(f"Class {class_name} score: {score}")

print(f"Final score: {final_score}")

Class Fe score: 0.9999999999999999
Class Zn score: 1.0
Class B score: 1.0
Class Cu score: 0.9999999999999994
Class S score: 0.9999999999999998
Class Mn score: 1.0000000000000002
Final score: 0.9999999999999999


# EBM

In [10]:
from interpret.glassbox import ExplainableBoostingRegressor
from MultiOutputTuner import MultiOutputTuner
from joblib import parallel_backend

# TODO: Too slow to train
param_grid = {
    "model__max_bins": [64, 128, 256],
    "model__learning_rate": [0.01, 0.04, 0.004],
    "model__max_leaves": [2, 3, 5],
    # "model__min_samples_leaf": [2, 3, 5],
    # "model__interactions": [20, 30, 40],
}

model = MultiOutputTuner(
    ExplainableBoostingRegressor(
        random_state = 42),
    param_grid,
    scoring="neg_mean_squared_error",
    cv=5,
    verbose=1,
    reduce=0.95
)

start = time.perf_counter()
with parallel_backend("threading"):
    model.fit(X_train, y_train)
end = time.perf_counter()
print(f"Train time: {end - start:4f}")

predictions = model.predict(X_test)
mse = np.mean((y_test - predictions) ** 2, axis=0)
print("MSE per class:", mse)
model.save("models/hsi/ebm_95")

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Column: 2 	 score: -0.5134736805868949 	 params: {'model__learning_rate': 0.01, 'model__max_bins': 256, 'model__max_leaves': 2, 'model__min_samples_leaf': 2}
Column: 4 	 score: -0.7320613299492456 	 params: {'model__learning_rate': 0.04, 'model__max_bins': 256, 'model__max_leaves': 3}
Column: 3 	 score: -0.9013286101931431 	 params: {'model__interactions': 20, 'model__learning_rate': 0.04, 'model__max_bins': 128, 'model__max_leaves': 3, 'model__min_samples_leaf': 5}
Column: 3 	 score: -0.9074534146345649 	 params: {'model__learning_rate': 0.04, 'model__max_bins': 128, 'model__max_leaves': 3}
Column: 5 	 score: -0.7225166

# RegressorTree

In [16]:
from interpret.glassbox import RegressionTree

param_grid = {
    "max_depth": [2, 3, 4, 5, 6]
}

models = []
predictions = np.zeros_like(y_test)
# model = MultiOutputTuner(
#     RegressionTree(random_state = 42),
#     param_grid,
#     scoring="neg_mean_squared_error",
#     verbose = 1
# )

start = time.perf_counter()
for i, col in enumerate(column_names):
    model = RegressionTree(random_state=42)
    model.fit(X_train, y_train[:,i])
    models.append(model)
    predictions[:,i] = model.predict(X_test)
end = time.perf_counter()
print(f"Train time: {end - start:4f}")

mse = np.mean((y_test - predictions) ** 2, axis=0)
print("MSE per class:", mse)

Train time: 0.031637
MSE per class: [0.68311472 0.74734655 0.71749339 0.88097142 0.89095415 0.69788554]


In [ ]:
show(

# LinearRegression

In [13]:
from interpret.glassbox import LinearRegression

models = []
predictions = np.zeros_like(y_test)

start = time.perf_counter()
for i, col in enumerate(column_names):
    model = LinearRegression()
    model.fit(X_train, y_train[:,i ])

    predictions[:, i] = model.predict(X_test)
    models.append(model)
end = time.perf_counter()
print(f"Train time: {end - start:4f}")

mse = np.mean((y_test - predictions) ** 2, axis=0)
print("MSE per class:", mse)

Train time: 0.623400
MSE per class: [0.69618927 0.68608632 0.67934925 0.79644612 0.78763307 0.62539252]
